# Municipal 311 Assistant — Adapter Sprint (QLoRA + DPO under a 15GB ceiling)

Rewrites messy resident 311 complaints into structured dispatch tickets. QLoRA SFT, a deliberate
OOM + fix, and two DPO passes were all actually run on a Colab T4; every number in this notebook is
loaded from a tracked evidence file in `data/` or `logs/`, not typed in by hand.

**Headline result**: DPO v1 (150 missing-address-only preference pairs) overgeneralized into "always
output null" -- it broke perfectly correct addressed-ticket behavior that QLoRA SFT alone already had
(`addressed_control_accuracy` dropped from 1.0 to 0.0 on the deterministic held-out comparison). A
balanced DPO v2 preference dataset (missing-address *and* addressed-control pairs) fixed the
regression while keeping the missing-address behavior intact. That failure-and-fix is the actual
experimental finding here -- see §7 for the full comparison and its correct interpretation.

This notebook documents the pipeline in the order it was actually run. Code cells that call training
scripts are shown for reproducibility (see `README.md`'s "How to run" section) -- they are **not**
re-executed by the cells that follow; the results sections load the real tracked output instead.

## 0. Setup

In [ ]:
!pip install -q -U -r requirements.txt


In [ ]:
import json, os

def show(path):
    with open(path) as f:
        data = json.load(f)
    print(f"--- {path} ---")
    print(json.dumps(data, indent=2))
    return data

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
QLORA_CONFIG_PATH = "configs/qlora_config.json"
DPO_CONFIG_PATH = "configs/dpo_config.json"


## 1. SFT instruction data (360 examples, offline generator)

`src/generate_instruction_data.py` produced 360 synthetic complaint -> ticket examples, split
280/30/50 train/val/test (seed 42). No external API calls.

In [ ]:
!python src/generate_instruction_data.py --n 360 --seed 42 --out-dir data


In [ ]:
show("data/generation_summary.json")


## 2. Token-length audit -> max_seq_len = 256 (approved)

`src/audit_instruction_data.py` measured the real `Qwen/Qwen2.5-0.5B-Instruct` tokenizer against all
280 training examples. 256 preserves every example's complete assistant target (0 truncated), vs.
142/280 truncated at 192 and 280/280 at 128. This value is written into
`configs/qlora_config.json`'s `max_seq_len`.

In [ ]:
!python src/audit_instruction_data.py --data data/sft_train.jsonl --model {MODEL_NAME} --out data/token_length_audit.json


In [ ]:
audit = show("data/token_length_audit.json")
print()
print("Approved max_seq_len:", json.load(open(QLORA_CONFIG_PATH))["max_seq_len"])


## 3. QLoRA supervised fine-tuning

LoRA r=16, alpha=32, dropout=0.05, target modules `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj,
down_proj` (verified against the model's actual linear layers at load time, not assumed). 4-bit NF4
base with double quantization. `validate_labels()` aborts before training if any example has zero
non-masked (`-100`) label tokens -- prompt tokens are always masked out of the loss.

**Note on losses**: `train_qlora.py` does not save `trainer.state.log_history`, so the final training
loss for this run was not persisted to a tracked file -- this notebook does not report a QLoRA loss
number for that reason (see the README's rubric-gap note), only the real wall-clock/memory
measurements below.

In [ ]:
!python src/train_qlora.py --config {QLORA_CONFIG_PATH} --data-dir data --out-dir outputs --log-path logs/experiment_log_qlora_final.json


In [ ]:
record = show("logs/experiment_log_qlora_final.json")[0]
print()
print(f"Trainable params: {record['config']['trainable_params']:,} / {record['config']['total_params']:,} ({record['config']['trainable_pct']}%)")
print(f"Wall time: {record['wall_time_sec']} s   Avg step time: {record['avg_step_time_sec']} s")
print(f"Peak allocated: {record['peak_allocated_gb']} GB   Peak reserved: {record['peak_reserved_gb']} GB")


## 4. Deliberate OOM, diagnosis, and the two-cheapest-fixes recovery

Unsafe config (genuinely exceeds T4 memory): micro-batch 32, sequence length 1024, gradient
checkpointing **off**, on a freshly loaded quantized base model.

Recovery changes **only two variables**: micro-batch 32 -> 4, and gradient checkpointing off -> on.
Sequence length stays at 1024 (unchanged from the unsafe attempt) so the recovery isolates which of
the two cheap fixes resolves the OOM, rather than changing three variables at once. Before the
recovery step, an entirely fresh quantized base model and a fresh LoRA adapter are reloaded -- the
recovery never resumes the base model the failed attempt already mutated.

In [ ]:
!python src/train_qlora.py --config {QLORA_CONFIG_PATH} --data-dir data --out-dir outputs --log-path logs/experiment_log_oom_final.json --run-oom-demo


In [ ]:
records = show("logs/experiment_log_oom_final.json")
oom_attempt, post_fix = records[0], records[1]
print()
print(f"Unsafe attempt: status={oom_attempt['status']} exception={oom_attempt['exception_type']} "
      f"peak_allocated={oom_attempt['peak_allocated_gb']}GB peak_reserved={oom_attempt['peak_reserved_gb']}GB "
      f"wall_time={oom_attempt['wall_time_sec']}s")
print(f"Recovery:       status={post_fix['status']} "
      f"peak_allocated={post_fix['peak_allocated_gb']}GB peak_reserved={post_fix['peak_reserved_gb']}GB "
      f"wall_time={post_fix['wall_time_sec']}s")

with open("logs/oom_traceback.txt") as f:
    print()
    print("Real captured traceback (last 500 chars):")
    print(f.read()[-500:])


## 5. DPO v1 -- missing-address-only preference data

`src/generate_address_dpo_data.py` built 150 pairs, all missing-address complaints: `chosen`
preserves `address_or_null: null`, `rejected` is byte-for-byte identical except `address_or_null` is
a fabricated street.

**Reference-policy design**: the SFT adapter (`outputs/qlora_adapter`) loads once as `PeftModel`'s
normal `"default"` adapter. `train_dpo.py` passes `ref_model=None`; the installed `trl 1.14.0`
`DPOTrainer` creates a second `"ref"` adapter automatically inside its own constructor and copies
every `.default.` parameter into `.ref.` -- an exact frozen copy of the SFT adapter.
`verify_reference_setup()` checks this rather than assuming it.

**Real compatibility failure + fix**: the first real run crashed on its first optimizer step
(`NotImplementedError: _amp_foreach_non_finite_check_and_unscale_cuda ... not implemented for
'BFloat16'`) because TRL's constructor casts the trainable `"default"` adapter to bfloat16 right
after copying it into `"ref"`, and this T4's `fp16=True` GradScaler can't unscale BF16 gradients.
`restore_fp32_policy_for_fp16_t4()` replaces each trainable `.default.` tensor with an FP32 clone of
its still-FP32 `.ref.` counterpart before training starts.

In [ ]:
!python src/generate_address_dpo_data.py --seed 101
!python src/audit_address_dpo_data.py --pairs data/dpo_pairs.jsonl --out data/dpo_pairs_audit.json


In [ ]:
audit_v1 = show("data/dpo_pairs_audit.json")


In [ ]:
!python src/train_dpo.py --dpo-config {DPO_CONFIG_PATH} --qlora-config {QLORA_CONFIG_PATH} \
  --data data/dpo_pairs.jsonl --out-dir outputs --log-path logs/experiment_log_dpo_v1_final.json \
  --log-history-path logs/dpo_trainer_v1_log_history.json


In [ ]:
record = show("logs/experiment_log_dpo_v1_final.json")[0]
log_history = show("logs/dpo_trainer_v1_log_history.json")
reward_entry = [e for e in log_history if "rewards/margins" in e][0]
print()
print(f"Wall time: {record['wall_time_sec']}s  Avg step time: {record['avg_step_time_sec']}s")
print(f"Peak allocated: {record['peak_allocated_gb']}GB  Peak reserved: {record['peak_reserved_gb']}GB")
print(f"rewards/margins: {reward_entry['rewards/margins']}  rewards/accuracies: {reward_entry['rewards/accuracies']}")


### Real held-out failure

`logs/eval_comparison_dpo_v1.json` (deterministic 10-example comparison, see §7 for the full table):
DPO v1 scored `missing_address_accuracy = 1.0` but **`addressed_control_accuracy = 0.0`** -- it output
`address_or_null: null` on all 5 addressed-control examples too, not just the 5 null-gold ones. With
150/150 preference pairs all showing "prefer null," DPO had no counter-signal and overgeneralized.

## 6. DPO v2 -- balanced preference data (the fix)

`src/generate_address_dpo_data_v2.py` built a balanced 150-pair replacement: 75 missing-address pairs
(same design as v1) plus 75 new addressed-control pairs, where `chosen` preserves a real,
complaint-stated street and `rejected` is the same ticket with `address_or_null` set to `null` -- the
mirror-image corruption. Every complaint was freshly generated and checked for zero overlap with
`sft_train`/`sft_val`/`sft_test` **and** with DPO v1's own prompts. Same DPO training configuration as
v1 (`beta`, `max_length`, batch size, gradient accumulation, epochs, learning rate, optimizer,
precision, gradient checkpointing, seed) -- only the preference data changed.

In [ ]:
!python src/generate_address_dpo_data_v2.py --seed 42


In [ ]:
summary_v2 = show("data/dpo_pairs_v2_summary.json")


In [ ]:
!python src/audit_address_dpo_data_v2.py --pairs data/dpo_pairs_v2.jsonl --model {MODEL_NAME} --out data/dpo_pairs_v2_audit.json


In [ ]:
audit_v2 = show("data/dpo_pairs_v2_audit.json")
print()
print("Real token-length check:", audit_v2["max_length_check"])


In [ ]:
!python src/train_dpo.py --dpo-config {DPO_CONFIG_PATH} --qlora-config {QLORA_CONFIG_PATH} \
  --data data/dpo_pairs_v2.jsonl --out-dir outputs --adapter-out-dir outputs/dpo_adapter_v2 \
  --log-path logs/experiment_log_dpo_v2_final.json --log-history-path logs/dpo_trainer_v2_log_history.json


In [ ]:
record = show("logs/experiment_log_dpo_v2_final.json")[0]
log_history = show("logs/dpo_trainer_v2_log_history.json")
reward_entry = [e for e in log_history if "rewards/margins" in e][0]
print()
print(f"Wall time: {record['wall_time_sec']}s  Avg step time: {record['avg_step_time_sec']}s")
print(f"Peak allocated: {record['peak_allocated_gb']}GB  Peak reserved: {record['peak_reserved_gb']}GB")
print(f"rewards/margins: {reward_entry['rewards/margins']}  rewards/accuracies: {reward_entry['rewards/accuracies']}")


## 7. Final deterministic held-out comparison: Base vs. QLoRA vs. DPO v1 vs. DPO v2

Same 10 held-out complaints for every model (5 gold-null-address + 5 gold-addressed, selected
deterministically before any inference -- `src/evaluate.py::select_comparison_examples`). Loaded here
directly from the tracked comparison artifacts -- nothing recomputed.

In [ ]:
v1_comparison = show("logs/eval_comparison_dpo_v1.json")
v2_comparison = show("logs/eval_comparison_dpo_v2.json")

assert v1_comparison["selection_rule"]["selected_test_indices"] == v2_comparison["selection_rule"]["selected_test_indices"], \
    "both comparisons must use the identical fixed 10 test indices"
print("Selected test indices (identical across both runs):", v1_comparison["selection_rule"]["selected_test_indices"])


In [ ]:
import pandas as pd

metrics = {
    "Base": v1_comparison["metrics"]["base"],
    "QLoRA": v1_comparison["metrics"]["qlora"],
    "DPO v1": v1_comparison["metrics"]["dpo"],
    "DPO v2": v2_comparison["metrics"]["dpo"],
}

# Sanity check: base/qlora must be identical between the two comparison runs (determinism check)
assert v1_comparison["metrics"]["base"] == v2_comparison["metrics"]["base"]
assert v1_comparison["metrics"]["qlora"] == v2_comparison["metrics"]["qlora"]

rows = ["json_valid_rate", "field_complete_rate", "category_accuracy", "address_exact_match_accuracy",
        "missing_address_accuracy", "hallucinated_address_count", "addressed_control_accuracy"]
table = pd.DataFrame({name: {r: m[r] for r in rows} for name, m in metrics.items()})
table


### Interpretation -- read this carefully

- QLoRA SFT alone already reaches perfect scores on every metric on this 10-example set, and DPO v2
  also reaches perfect scores. **This is not evidence that DPO v2 outperformed QLoRA** -- there is no
  headroom left to show improvement over on this small, fixed diagnostic set; QLoRA got there first.
- The real, demonstrated result is that **DPO v1 introduced a regression**
  (`addressed_control_accuracy` 0.0, down from QLoRA's 1.0) by overgeneralizing a 100%-missing-address
  preference set into "always output null," and **the balanced DPO v2 preference set eliminated that
  regression while preserving the missing-address behavior** (`missing_address_accuracy` stayed 1.0 in
  both v1 and v2). The finding is about preference-dataset design causing, and then fixing, a specific
  behavioral failure -- not about DPO improving on QLoRA in general.
- 10 examples is a small, fixed diagnostic set built to directly probe one behavior (missing- vs.
  addressed-address handling), not to support a general claim about overall model quality.

## 8. AI collaboration

This project was built through iterative, Claude-assisted development, with the user driving every
experimental decision and every real Colab run. Claude's role was code, design, and diagnostic work
against evidence the user reported back from actual runs -- not independent experimentation.
Claude-assisted work included: the synthetic instruction-dataset generator and its later revision
after manual review found template/priority mismatches; the token-length audit script (revised twice
as new requirements -- per-candidate truncation, then real DPO prompt/combined lengths -- were
specified); the FP16-not-BF16 decision and the `check_hardware()` guard for the T4's lack of native
BF16; a series of real-API-verified compatibility fixes (`eval_strategy` vs. `evaluation_strategy`,
`dtype` vs. deprecated `torch_dtype`, `gradient_checkpointing_kwargs`, and the
`warmup_ratio`->`warmup_steps` conversion after the user reported the exact installed
`transformers 5.16.1` behavior); the `GpuMemoryMonitorCallback` memory-ceiling instrumentation; the
deliberate-OOM/two-cheapest-fixes recovery design; investigating and implementing the TRL
`"default"`/`"ref"` automatic reference-adapter behavior after the user reported the exact installed
`trl`/`peft` API surface; the FP32-policy restoration fix after the user reported the real
`NotImplementedError` from a live Colab run; the deterministic 10-example held-out evaluation design
(fixed selection rule, sequential model load/unload); diagnosing the DPO v1 addressed-control
regression from the real comparison JSON the user reported back; and designing the balanced DPO v2
preference dataset as the fix.

**What Claude did not do**: run any cell in this notebook, observe any GPU, or produce any of the
numbers shown in §§1-7. Every metric, traceback, and reward value here was reported back by the user
from an actual Colab execution and is stored in the tracked files this notebook loads. Where a number
could not be found in a tracked file (the QLoRA final training loss -- see §3), it is reported as
missing, not filled in from memory of an earlier conversation.

## 9. Reproducibility notes

- Every cell above that calls a training/generation script documents the exact command that produced
  the real evidence loaded afterward -- it does not need to be rerun to see the results.
- `outputs/` (model checkpoints and adapter weights: `outputs/qlora_adapter`, `outputs/dpo_adapter`,
  `outputs/dpo_adapter_v2`) is **gitignored** -- not present in GitHub. Rerunning the training cells
  above regenerates it locally; it is not something a reader can fetch from the repo.
- `data/` and `logs/` are committed in full and are the actual source of truth for every number in
  this notebook and in `README.md`.
- The `deliberate_oom` config block's `unsafe_batch_size`/`unsafe_seq_len` may need to be raised
  further on a different Colab GPU allocation to reproduce a genuine OOM -- the goal is a real
  `torch.cuda.OutOfMemoryError`, not these exact numbers.